In [ ]:
import os
import shutil
from functools import partial
from pathlib import Path

from urbanopt_des.uo_cli_wrapper import UOCliWrapper
from tools.docker_uo_cli_wrapper import DockerUOCliWrapper
from geojson_modelica_translator.modelica.modelica_runner import ModelicaRunner

# -- Execution mode -----------------------------------------------------------
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
DOCKER_IMAGE_TAG = "urbanopt-cli:1.2-ubuntu22"
# ---------------------------------------------------------------------------

if USE_DOCKER:
    UO = partial(DockerUOCliWrapper, image_tag=DOCKER_IMAGE_TAG)
else:
    UO = UOCliWrapper

# Autoreload dependencies while iterating on wrapper code.
%load_ext autoreload
%autoreload 2

In [ ]:
# Get the working directories
workdir = Path().resolve()
print(f"Working directory: {workdir}")

analysis_dir = workdir / "esbe_2026"
if not analysis_dir.exists():
    analysis_dir.mkdir()

template_data_dir = workdir / "data" / "templates"

print(f"template_data_dir: {template_data_dir}")
print(f"analysis_dir: {analysis_dir}")

# Get the number of usable cores for parallel processing by looking at the system (n-2)
num_usable_cores = max(1, (os.cpu_count() or 2) - 2)
print(f"Number of usable cores for parallel processing: {num_usable_cores}")

# update weather flag
update_weather_location = False
new_weather_epw = "USA_FL_MacDill.AFB.747880_TMY3"
new_weather_climate_zone = "1A"

# new_weather_epw = "Lecco_IT_TMY"
# new_weather_climate_zone = "4A"

### Activity 4: DES/TEN

In [ ]:
# This is the same setup pattern as prior activities, but in a new directory
activity_4_analysis_dir = analysis_dir / "activity_4"
if not activity_4_analysis_dir.exists():
    activity_4_analysis_dir.mkdir()

uo_coincident = UO(
    activity_4_analysis_dir, "coincident_pre", template_dir=template_data_dir
)

uo_coincident.create_example_coincident_project()

uo_coincident.create_scenarios("class_project_coincident.json")

# Copy over to the coincident directory for activity 4
uo_coincident = uo_coincident.update_project_files("coincident")

# Run sims faster
uo_coincident.set_number_parallel(num_usable_cores)

# Copy over the weather files
uo_coincident.copy_over_weather()

# Change weather file in feature + mapper files when enabled
if update_weather_location:
    uo_coincident.replace_weather_file_in_feature_and_mapper_file(
        new_weather_epw, new_weather_climate_zone
    )

# Fix issues -- copy over the updated Baseline.rb mapper file
shutil.copy2(
    template_data_dir / "mappers" / "Baseline.rb",
    uo_coincident.project_path / "mappers" / "Baseline.rb",
)
# Copy over the base workflow from the template. This one includes all the EEMs that are needed in the class project since
# the class_project_workflow.json inherits from base_workflow.osw. This is a workaround to avoid having to patch the workflow file to add in the missing EEM steps.
shutil.copy2(
    template_data_dir / "mappers" / "base_workflow.osw",
    uo_coincident.project_path / "mappers" / "base_workflow.osw",
)

In [ ]:
# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

# Post-process and visualize the baseline
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_feature("class_project_coincident.json")

In [ ]:
scenario_path = uo_coincident.project_path / "baseline_scenario.csv"
feature_path = uo_coincident.project_path / "class_project_coincident.json"
sys_param_path = uo_coincident.project_path / "sys_params.json"

# save these as the relative paths because they will be run from within docker

print(f"scenario_path: {scenario_path}")
print(f"feature_path: {feature_path}")
print(f"sys_param_path: {sys_param_path}")

# Now create the 4G system
uo_coincident.des_params(scenario_path, feature_path, sys_param_path)

In [ ]:
# fix values in the sys_params.json file for the 4G system
# "central_cooling_plant_parameters": {
#   "heat_flow_nominal": 8000000,        // 8 MW, was 7999 W
#   "cooling_tower_fan_power_nominal": 50000,
#   "mass_chw_flow_nominal": 85,         // ~Σ building chw flows
#   "chiller_water_flow_minimum": 20,    // ~0.25 × nominal
#   "mass_cw_flow_nominal": 100,         // ~1.15 × chw nominal — this prevents the freeze
#   ...
# }

# create the 4g model
uo_coincident.des_create(sys_param_path, feature_path, "four_g")

In [ ]:
# Run the 4G system with the Modelica runner for now, even
# though the URBANopt-DES package also has a run command.

# Run the original 4G model from the uo CLI flow
mr = ModelicaRunner()

des_scenario_name = "four_g"
des_analysis_4g_dir = activity_4_analysis_dir / des_scenario_name

# Print the full arguments so we can verify what is being set
print(f"{des_scenario_name}.Districts.district")
print(f"file_to_load={des_analysis_4g_dir / 'package.mo'}")
print(f"run_path={des_analysis_4g_dir}")

print(f"Running model in the following directory: {des_analysis_4g_dir}")

# Check whether results from a previous run already exist
results_path = (
    des_analysis_4g_dir / f"{des_scenario_name}.Districts.DistrictEnergySystem_results"
)
if results_path.exists():
    print("Results already exist, skipping model run.")
else:
    # Use the gmt-om-runner to run the model, not the UO DES cli
    _, results_path = mr.run_in_docker(
        "compile_and_run",
        f"{des_scenario_name}.Districts.DistrictEnergySystem",
        file_to_load=f"{des_analysis_4g_dir / 'package.mo'}",
        run_path=des_analysis_4g_dir,
        # march 1 -> march 14
        start_time=5097600,
        stop_time=6307200,
    )

# running 4G model for 2 weeks takes ~

In [ ]:
# create the 5G model
sys_param_path_5g = uo_coincident.project_path / "sys_params_5g.json"
uo_coincident.des_create(sys_param_path_5g, feature_path, "five_g")

In [ ]:
# Run the 5G system with the Modelica runner for now, even
# though the URBANopt-DES package also has a run command.

# Run the original 5G model from the uo CLI flow
mr = ModelicaRunner()

des_scenario_name = "five_g"
des_analysis_5g_dir = activity_4_analysis_dir / des_scenario_name

# Print the full arguments so we can verify what is being set
print(f"{des_scenario_name}.Districts.district")
print(f"file_to_load={des_analysis_5g_dir / 'package.mo'}")
print(f"run_path={des_analysis_5g_dir}")

print(f"Running model in the following directory: {des_analysis_5g_dir}")

# Check whether results from a previous run already exist
results_path = (
    des_analysis_5g_dir / f"{des_scenario_name}.Districts.DistrictEnergySystem_results"
)
if results_path.exists():
    print("Results already exist, skipping model run.")
else:
    # Use the gmt-om-runner to run the model, not the UO DES cli
    _, results_path = mr.run_in_docker(
        "compile_and_run",
        f"{des_scenario_name}.Districts.DistrictEnergySystem",
        file_to_load=f"{des_analysis_5g_dir / 'package.mo'}",
        run_path=des_analysis_5g_dir,
        # march 1 -> march 14
        start_time=5097600,
        stop_time=6307200,
    )

# running 4G model for 2 weeks takes ~